# Pjesa 4B — Sisteme të tjera me dinamikë kaotike

Ky notebook shërben si **zgjerim** i materialit mbi kaosin, përtej dy shembujve klasikë të diskutuar zakonisht në hyrje: **harta logjistike** dhe **sistemi i Lorencit**.  
Qëllimi është të paraqiten disa sisteme të tjera, po aq të rëndësishme, që shfaqin:

- ndjeshmëri ndaj kushteve fillestare,
- tërheqës të çuditshëm,
- bifurkacione,
- ndërthurje mes rendit dhe çrregullsisë,
- dhe një strukturë gjeometrike shumë të pasur në hapësirën e fazës.

Notebook-u është menduar në të njëjtin nivel me materialin kryesor të klasës:  
**teori e shpjeguar qartë, formulim matematikor, implementim numerik dhe interpretim fizik.**

---

## Sistemet që do të studiojmë

Në vijim do të shqyrtojmë pesë shembuj të rëndësishëm:

1. **Sistemi i Rössler-it** — një sistem ODE me tre variabla, me tërheqës kaotik të thjeshtë në dukje, por shumë të pasur.
2. **Oscilatori i detyruar i Duffing-ut** — shembull i rëndësishëm i kaosit në mekanikë jolineare.
3. **Harta e Hénon-it** — një hartë diskrete dypërmasore me tërheqës fraktal klasik.
4. **Harta standarde (kicked rotor)** — shembull themelor në dinamikën Hamiltoniane dhe tranzicionin drejt kaosit.
5. **Qarku i Chua-s** — një sistem elektronik jolinear që prodhon një prej tërheqësve më të famshëm kaotikë.

Në fund do të bëjmë edhe një **krahasim konceptual** mes këtyre sistemeve.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True

def rk4_step(f, t, y, dt, **params):
    k1 = f(t, y, **params)
    k2 = f(t + dt/2, y + dt*k1/2, **params)
    k3 = f(t + dt/2, y + dt*k2/2, **params)
    k4 = f(t + dt, y + dt*k3, **params)
    return y + dt*(k1 + 2*k2 + 2*k3 + k4)/6

def integrate_ode(f, y0, t0, t1, dt, **params):
    n = int(np.ceil((t1 - t0)/dt))
    t = np.linspace(t0, t0 + n*dt, n+1)
    y = np.zeros((n+1, len(y0)), dtype=float)
    y[0] = y0
    for i in range(n):
        y[i+1] = rk4_step(f, t[i], y[i], dt, **params)
    return t, y

def plot_two_series(t, y1, y2, labels=("Trajektorja 1", "Trajektorja 2"), title=""):
    plt.figure(figsize=(9, 4.5))
    plt.plot(t, y1, label=labels[0], linewidth=1.6)
    plt.plot(t, y2, "--", label=labels[1], linewidth=1.2)
    plt.xlabel("Koha")
    plt.ylabel("Amplituda")
    plt.title(title)
    plt.legend()
    plt.show()

def divergence_curve(a, b, eps=1e-16):
    d = np.linalg.norm(a - b, axis=1)
    return np.maximum(d, eps)

def wrap_angle(theta):
    return (theta + np.pi) % (2*np.pi) - np.pi


## 1. Vështrim i përgjithshëm: pse na duhen shembuj të tjerë?

Lorenci është shembulli më i famshëm i një sistemi kaotik të vazhdueshëm, por në praktikë dhe në teori ekzistojnë shumë klasa të ndryshme sistemesh jolineare që gjenerojnë kaos. Këto klasa ndryshojnë sipas:

- **natyrës së kohës**: sistem i vazhdueshëm (ODE) kundrejt harte diskrete,
- **strukturës dinamike**: disipativ kundrejt Hamiltonian,
- **origjinës fizike**: mekanikë, elektronikë, dinamika e popullatave, rrjedhje fluidesh, etj.,
- **llojit të objektit gjeometrik** që organizon dinamikën: orbitë periodike, torus, tërheqës i çuditshëm, ishuj stabiliteti, shtresë kaotike.

Kjo shumëllojshmëri është e rëndësishme pedagogjikisht, sepse i ndihmon studentët të kuptojnë se **kaosi nuk është një fenomen i izoluar**, por një tipar i përgjithshëm i sistemeve jolineare.

Në këtë kuptim, çdo shembull i mëposhtëm na tregon një “fytyrë” të ndryshme të kaosit.


## 2. Sistemi i Rössler-it

### 2.1. Përkufizimi

Sistemi i Rössler-it jepet nga ekuacionet:

$$
\dot{x} = -y - z,
$$
$$
\dot{y} = x + a y,
$$
$$
\dot{z} = b + z(x - c).
$$

Ky është një sistem me tre variabla, i ngjashëm në dimension me sistemin e Lorencit, por me një gjeometri shumë më “të hollë” në hapësirën e fazës. Për vlera të caktuara të parametrave, ai shfaq një tërheqës kaotik shumë karakteristik.

### 2.2. Interpretim cilësor

- Ekuacionet për \(x\) dhe \(y\) krijojnë një lëvizje rrotulluese pothuajse-planare.
- Variabla \(z\) ndërhyn si mekanizëm jolinear që, herë pas here, shtyn trajektoren larg rajonit të qetë të rrotullimit.
- Kështu krijohet një alternim mes **rrotullimit lokal** dhe **largimit global**, gjë që përbën bazën e dinamikës kaotike.

Zgjedhja klasike e parametrave është:

$$
a = 0.2, \qquad b = 0.2, \qquad c = 5.7.
$$


In [ ]:
def rossler(t, state, a, b, c):
    x, y, z = state
    dx = -y - z
    dy = x + a*y
    dz = b + z*(x - c)
    return np.array([dx, dy, dz], dtype=float)

a, b, c = 0.2, 0.2, 5.7
t, sol = integrate_ode(
    rossler,
    y0=np.array([0.1, 0.0, 0.0]),
    t0=0.0,
    t1=200.0,
    dt=0.02,
    a=a, b=b, c=c
)

# heqim tranzientin fillestar
mask = t > 50.0
x, y, z = sol[mask, 0], sol[mask, 1], sol[mask, 2]

from mpl_toolkits.mplot3d import Axes3D  # aktivizim 3D

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(x, y, z, linewidth=0.6)
ax.set_title("Tërheqësi i Rössler-it")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.show()


### 2.3. Koment mbi figurën

Në kontrast me Lorencin, ku tërheqësi ka dy “krahë” të dukshëm, tërheqësi i Rössler-it duket si një **spirale e shtypur** që hapet dhe paloset vazhdimisht. Kjo pamje është e dobishme për studentët, sepse e bën më të qartë idenë se kaosi mund të lindë nga një mekanizëm shumë i thjeshtë:

1. **rrotullim** në një zonë pothuajse të rregullt,
2. **zgjerim** i gabimeve të vogla,
3. **palosje** e trajektores për ta rikthyer përsëri në rajonin e kufizuar.

Ky kombinim “stretch-and-fold” është një motiv themelor në teorinë e kaosit.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(x, y, linewidth=0.7)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Projekcioni në planin (x, y) për sistemin e Rössler-it")
plt.axis("equal")
plt.show()


### 2.4. Ndjeshmëria ndaj kushteve fillestare

Si në sistemin e Lorencit, edhe këtu mund të krahasojmë dy trajektore që fillojnë shumë pranë njëra-tjetrës.


In [ ]:
t1, sol1 = integrate_ode(rossler, np.array([0.1, 0.0, 0.0]), 0.0, 120.0, 0.02, a=a, b=b, c=c)
t2, sol2 = integrate_ode(rossler, np.array([0.100001, 0.0, 0.0]), 0.0, 120.0, 0.02, a=a, b=b, c=c)

plt.figure(figsize=(9, 4.5))
plt.plot(t1, sol1[:,0], label="x(t), trajektorja 1", linewidth=1.4)
plt.plot(t2, sol2[:,0], "--", label="x(t), trajektorja 2", linewidth=1.2)
plt.xlabel("Koha")
plt.ylabel("x(t)")
plt.title("Ndjeshmëria ndaj kushteve fillestare në sistemin e Rössler-it")
plt.legend()
plt.show()

d = divergence_curve(sol1, sol2)
plt.figure(figsize=(8, 4.5))
plt.semilogy(t1, d)
plt.xlabel("Koha")
plt.ylabel("Distanca |Δy(t)|")
plt.title("Rritja e divergjencës mes dy trajektoreve")
plt.show()


Ky eksperiment tregon qartë se **parashikueshmëria afatgjatë humbet**, edhe kur ekuacionet janë krejtësisht deterministe.

---

## 3. Oscilatori i detyruar i Duffing-ut

### 3.1. Modeli

Oscilatori i Duffing-ut është një model klasik i një oshilatori jolinear me forcim kubik. Forma e tij e detyruar dhe e zbutur është:

$$
\ddot{x} + \delta \dot{x} + \alpha x + \beta x^3 = \gamma \cos(\omega t).
$$

Kjo është një barazim diferencial i rendit të dytë. Për ta integruar numerikisht, e kthejmë në sistem të rendit të parë duke vendosur:

$$
v = \dot{x}.
$$

Atëherë:

$$
\dot{x} = v,
$$
$$
\dot{v} = -\delta v - \alpha x - \beta x^3 + \gamma \cos(\omega t).
$$

### 3.2. Rëndësia fizike

Ky sistem shfaqet në shumë kontekste:

- oscilatorë mekanikë me elasticitet jolinear,
- qarqe elektrike jolineare,
- rezonanca jo-lineare në struktura,
- vibrime me shumë regjime të mundshme.

Kur potenciali

$$
V(x) = \frac{\alpha}{2}x^2 + \frac{\beta}{4}x^4
$$

ka formë “double well”, sistemi mund të kalojë në mënyrë të parregullt mes dy puseve të potencialit, duke gjeneruar kaos.


In [ ]:
def duffing(t, state, delta, alpha, beta, gamma, omega):
    x, v = state
    dx = v
    dv = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
    return np.array([dx, dv], dtype=float)

params_duff = dict(delta=0.2, alpha=-1.0, beta=1.0, gamma=0.3, omega=1.2)

t, sol = integrate_ode(
    duffing,
    y0=np.array([0.1, 0.0]),
    t0=0.0,
    t1=400.0,
    dt=0.01,
    **params_duff
)

x = sol[:,0]
v = sol[:,1]

plt.figure(figsize=(9, 4.5))
plt.plot(t[t <= 120], x[t <= 120], linewidth=1.0)
plt.xlabel("Koha")
plt.ylabel("x(t)")
plt.title("Oscilatori i Duffing-ut: sinjal kohor")
plt.show()

plt.figure(figsize=(7, 6))
plt.plot(x[5000:], v[5000:], linewidth=0.35)
plt.xlabel("x")
plt.ylabel("v")
plt.title("Portreti i fazës për oscilarorin e Duffing-ut")
plt.show()


### 3.3. Seksioni i Poincaré-së

Meqë sistemi është i detyruar periodikisht me frekuencë \(\omega\), është shumë e dobishme të vëzhgojmë gjendjen e sistemit vetëm në kohë të formës:

$$
t_n = nT, \qquad T = \frac{2\pi}{\omega}.
$$

Kjo quhet **seksion i Poincaré-së**. Në regjime periodike, pikat bien mbi një numër të kufizuar pikash. Në regjime kaotike, ato mbushin një strukturë të parregullt.


In [ ]:
T = 2*np.pi / params_duff["omega"]
n = np.arange(1, int(t[-1]/T))
sample_times = n*T

idx = np.searchsorted(t, sample_times)
idx = idx[idx < len(t)]

plt.figure(figsize=(7, 6))
plt.scatter(x[idx], v[idx], s=8)
plt.xlabel("x")
plt.ylabel("v")
plt.title("Seksioni i Poincaré-së për oscilarorin e Duffing-ut")
plt.show()


### 3.4. Ndjeshmëria ndaj kushteve fillestare

Për sisteme të detyruara si ky, dy trajektore shumë të afërta mund të ndjekin për një kohë të shkurtër njëri-tjetrin, por më pas të ndahen dukshëm. Kjo vërehet qartë edhe në ndryshimin e pikave të seksionit të Poincaré-së.


In [ ]:
t1, s1 = integrate_ode(duffing, np.array([0.1, 0.0]), 0.0, 200.0, 0.01, **params_duff)
t2, s2 = integrate_ode(duffing, np.array([0.100001, 0.0]), 0.0, 200.0, 0.01, **params_duff)

plt.figure(figsize=(9, 4.5))
plt.plot(t1[t1 <= 80], s1[t1 <= 80, 0], label="x(t), trajektorja 1", linewidth=1.4) # Increase from 80 to see the split between trajectories
plt.plot(t2[t2 <= 80], s2[t2 <= 80, 0], "--", label="x(t), trajektorja 2", linewidth=1.2) # Increase from 80 to see the split between trajectories
plt.xlabel("Koha")
plt.ylabel("x(t)")
plt.title("Ndjeshmëria ndaj kushteve fillestare për Duffing-un")
plt.legend()
plt.show()

---

## 4. Harta e Hénon-it

### 4.1. Përkufizimi

Harta e Hénon-it është një sistem diskret dypërmasor:

$$
x_{n+1} = 1 - a x_n^2 + y_n,
$$
$$
y_{n+1} = b x_n.
$$

Për parametrat klasikë

$$
a = 1.4, \qquad b = 0.3,
$$

kjo hartë prodhon një tërheqës fraktal të famshëm.

### 4.2. Pse është e rëndësishme?

Ky është një shembull shumë i mirë sepse:

- është **diskret**, jo i vazhdueshëm;
- është **shumë i thjeshtë për t’u implementuar**;
- megjithatë prodhon një strukturë gjeometrike jashtëzakonisht të pasur;
- ilustron qartë se kaosi nuk kërkon medoemos ekuacione diferenciale të komplikuara.


In [ ]:
def henon_map(x, y, a=1.4, b=0.3):
    xn = 1 - a*x*x + y
    yn = b*x
    return xn, yn

def iterate_henon(x0=0.1, y0=0.1, n=20000, a=1.4, b=0.3):
    xs = np.zeros(n)
    ys = np.zeros(n)
    x, y = x0, y0
    for i in range(n):
        x, y = henon_map(x, y, a=a, b=b)
        xs[i], ys[i] = x, y
    return xs, ys

xs, ys = iterate_henon()

plt.figure(figsize=(7, 6))
plt.scatter(xs[1000:], ys[1000:], s=0.2)
plt.xlabel("$x_n$")
plt.ylabel("$y_n$")
plt.title("Tërheqësi i Hénon-it")
plt.show()


### 4.3. Koment gjeometrik

Në pamje të parë, tërheqësi i Hénon-it duket si një re pikash. Por në zmadhim, vihet re se ai ka një **strukturë të palosur në shtresa shumë të holla**, tipike për një objekt fraktal.

Kjo është një mundësi shumë e mirë për të diskutuar me studentët idenë se:

- trajektorja nuk mbush plotësisht një zonë të zakonshme të planit;
- as nuk kufizohet në një kurbë të vetme të lëmuar;
- ajo jeton në një objekt me dimension më kompleks.


In [ ]:
xs2, ys2 = iterate_henon(x0=0.100001, y0=0.1)

nplot = 80
plt.figure(figsize=(8, 4.5))
plt.plot(xs[:nplot], label="x_n, trajektorja 1", marker="o", markersize=3)
plt.plot(xs2[:nplot], "--", label="x_n, trajektorja 2", marker="s", markersize=2)
plt.xlabel("n")
plt.ylabel("$x_n$")
plt.title("Dy trajektore shumë pranë në hartën e Hénon-it")
plt.legend()
plt.show()

d = np.sqrt((xs - xs2)**2 + (ys - ys2)**2)
plt.figure(figsize=(8, 4.5))
plt.semilogy(d[:400])
plt.xlabel("n")
plt.ylabel("Distanca")
plt.title("Divergjenca e trajektoreve në hartën e Hénon-it")
plt.show()


Këtu vërehet shumë qartë natyra e sistemeve diskrete kaotike: një ndryshim tepër i vogël në gjendjen fillestare amplifikohet me kalimin e iterimeve.

---

## 5. Harta standarde (kicked rotor)

### 5.1. Përkufizimi

Harta standarde, e njohur edhe si **Chirikov–Taylor map**, jepet nga:

$$
p_{n+1} = p_n + K \sin(\theta_n),
$$
$$
\theta_{n+1} = \theta_n + p_{n+1}.
$$

Në praktikë, këndi \(\theta\) reduktohet modulo \(2\pi\).

### 5.2. Kuptimi fizik

Kjo hartë lidhet me një **rotor të goditur periodikisht** dhe është një model shumë i rëndësishëm në dinamikën Hamiltoniane. Ajo është thelbësore sepse:

- tregon si shkatërrohen toruset invariante kur rritet parametri \(K\),
- shfaq bashkëjetesë mes zonave të rregullta dhe zonave kaotike,
- është shembull tipik i tranzicionit drejt kaosit në sisteme pothuajse-integrues.

Ndryshe nga sistemet disipative, këtu faza nuk “bie” mbi një tërheqës të çuditshëm në të njëjtin kuptim; përkundrazi, kemi një gjeometri të përzier në hapësirën e fazës.


In [ ]:
def standard_map(theta0, p0, K=1.2, n=2000):
    theta = np.zeros(n)
    p = np.zeros(n)
    theta[0], p[0] = theta0, p0
    for i in range(n-1):
        p[i+1] = p[i] + K*np.sin(theta[i])
        theta[i+1] = wrap_angle(theta[i] + p[i+1])
    return theta, p

plt.figure(figsize=(8, 6))
for th0 in np.linspace(-np.pi, np.pi, 18):
    for p0 in np.linspace(-2.5, 2.5, 7):
        th, pp = standard_map(th0, p0, K=1.2, n=900)
        plt.plot(th, pp, ".", markersize=0.7)
plt.xlabel(r"$\theta$")
plt.ylabel("p")
plt.title("Harta standarde për K = 1.2")
plt.show()


### 5.3. Interpretim

Në figurë mund të dallohen:

- **ishuj të rregullt** ku orbitat qëndrojnë mbi kurba të mbyllura,
- **zona të shpërndara** ku pika të shumta mbushin rajone më të çrregullta,
- kufij të ndërlikuar mes këtyre regjimeve.

Kjo është shumë e rëndësishme teorikisht: jo çdo sistem kaotik është thjesht “plotësisht kaotik”. Shpesh, veçanërisht në sistemet Hamiltoniane, kemi një **bashkëjetesë** të rendeve të ndryshme dinamike.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

for ax, K in zip(axes, [0.4, 1.8]):
    for th0 in np.linspace(-np.pi, np.pi, 16):
        for p0 in np.linspace(-2.5, 2.5, 6):
            th, pp = standard_map(th0, p0, K=K, n=700)
            ax.plot(th, pp, ".", markersize=0.6)
    ax.set_title(f"Harta standarde për K = {K}")
    ax.set_xlabel(r"$\theta$")
axes[0].set_ylabel("p")
plt.suptitle("Krahasim: regjim më i rregullt kundrejt regjim më kaotik")
plt.tight_layout()
plt.show()


Rritja e parametrës \(K\) zakonisht çon në zgjerim të rajoneve kaotike. Kjo është një mënyrë shumë vizuale për të diskutuar **bifurkacionet gjeometrike** në hapësirën e fazës.

---

## 6. Qarku i Chua-s

### 6.1. Modeli

Qarku i Chua-s është një nga sistemet më të famshme elektronike që prodhon kaos. Në formën e tij të thjeshtuar, ai mund të shkruhet si:

$$
\dot{x} = \alpha (y - x - h(x)),
$$
$$
\dot{y} = x - y + z,
$$
$$
\dot{z} = -\beta y,
$$

ku funksioni jolinear \(h(x)\) është zakonisht pjesor-linear:

$$
h(x) = m_1 x + \frac{1}{2}(m_0 - m_1)(|x+1| - |x-1|).
$$

### 6.2. Kuptimi fizik

Ky sistem është shumë i rëndësishëm sepse:

- jep një realizim eksperimental real të kaosit,
- tregon se kaosi nuk është vetëm abstraksion matematikor,
- lidh teorinë e sistemeve dinamike me qarqe elektrike reale.


In [ ]:
def chua_nonlinearity(x, m0, m1):
    return m1*x + 0.5*(m0 - m1)*(np.abs(x + 1) - np.abs(x - 1))

def chua(t, state, alpha, beta, m0, m1):
    x, y, z = state
    hx = chua_nonlinearity(x, m0, m1)
    dx = alpha*(y - x - hx)
    dy = x - y + z
    dz = -beta*y
    return np.array([dx, dy, dz], dtype=float)

params_chua = dict(alpha=15.6, beta=28.0, m0=-1.143, m1=-0.714)

t, sol = integrate_ode(
    chua,
    y0=np.array([0.1, 0.0, 0.0]),
    t0=0.0,
    t1=300.0,
    dt=0.01,
    **params_chua
)

mask = t > 80.0
x, y, z = sol[mask, 0], sol[mask, 1], sol[mask, 2]

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(x, y, z, linewidth=0.5)
ax.set_title("Tërheqësi i Chua-s")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.show()


### 6.3. Koment

Tërheqësi i Chua-s shpesh paraqitet si një strukturë me dy “fletë” apo dy “scrolls”, prandaj shpesh quhet **double-scroll attractor**. Ai është një nga shembujt më të njohur ku një sistem fizik elektronik i thjeshtë realizon në mënyrë të matshme një dinamikë kaotike.

Pedagogjikisht, ky shembull është shumë i vlefshëm, sepse studentët shohin se:

- kaosi mund të lindë nga **jo-linearitete shumë të thjeshta**,
- sistemet elektronike ofrojnë realizime laboratorike të drejtpërdrejta,
- analiza gjeometrike e tërheqësit mbetet po aq e rëndësishme sa analiza e sinjalit kohor.


In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(x, y, linewidth=0.4)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Projekcioni (x, y) për qarkun e Chua-s")
plt.show()


---

## 7. Krahasim konceptual mes sistemeve

Le të përmbledhim tiparet kryesore të pesë shembujve të studiuar.

### 7.1. Sisteme të vazhdueshme kundrejt hartave diskrete

- **Rössler**, **Duffing**, **Chua**: sisteme të vazhdueshme, të përshkruara nga ODE.
- **Hénon**, **standard map**: sisteme diskrete, të përshkruara nga iterime.

Kjo ndarje është shumë e rëndësishme. Në sistemet e vazhdueshme, trajektorja është kurbë në kohë; në sistemet diskrete, gjendja lëviz me hapa të dallueshëm.

### 7.2. Disipativ kundrejt Hamiltonian

- **Rössler**, **Duffing**, **Hénon**, **Chua** janë tipikisht shembuj disipativë: dinamika përqendrohet në rajone të kufizuara dhe shpesh bie mbi tërheqës të çuditshëm.
- **Harta standarde** është shembull Hamiltonian/ruajtës në kuptim gjeometrik: nuk kemi të njëjtin mekanizëm disipativ, por një mozaik të pasur me ishuj stabiliteti dhe rajone kaotike.

### 7.3. Gjeometria e kaosit

- **Rössler**: spirale që paloset.
- **Duffing**: strukturë e parregullt në portretin e fazës dhe seksionin e Poincaré-së.
- **Hénon**: tërheqës fraktal në plan.
- **Standard map**: bashkëjetesë e rendit dhe kaosit në hapësirën e fazës.
- **Chua**: tërheqës me dy-scroll.

### 7.4. Mesazhi i përgjithshëm

Një student që i kupton këto shembuj fiton një intuicion shumë më të gjerë për kaosin:

- kaosi nuk lidhet me një formulë të vetme;
- ai mund të shfaqet në mekanikë, elektronikë, dhe sisteme të pastra matematikore;
- analizat numerike dhe gjeometrike janë po aq të rëndësishme sa formulimi analitik.


## 8. Ushtrime të sugjeruara

1. **Rössler**  
   Ndryshoni parametrin \(c\) dhe vëzhgoni si ndryshon forma e tërheqësit.  
   A gjeni intervale ku dinamika duket më periodike?

2. **Duffing**  
   Ndryshoni amplitudën e forcimit \(\gamma\).  
   Si ndryshon seksioni i Poincaré-së?

3. **Hénon**  
   Provoni vlera të ndryshme të parametrës \(a\) dhe krahasoni densitetin e pikave në tërheqës.

4. **Harta standarde**  
   Krahasoni \(K = 0.2\), \(K = 0.8\), \(K = 1.5\), \(K = 3.0\).  
   Cila figurë ka më shumë zona të rregullta? Cila ka më shumë përzierje?

5. **Chua**  
   Ndryshoni pjerrësitë \(m_0\) dhe \(m_1\).  
   A ruhet struktura me dy-scroll apo sistemi kalon në regjime të tjera?

6. **Projekt i vogël**  
   Zgjidhni një nga sistemet e mësipërme dhe përpiloni:
   - ekuacionet,
   - kuptimin fizik,
   - metodën numerike,
   - figurat kryesore,
   - dhe një interpretim të qartë të kaosit të vërejtur.


## 9. Përfundime

Në këtë notebook u panë disa prej shembujve më të rëndësishëm të dinamikës kaotike përtej sistemit të Lorencit. Çdo sistem nxjerr në pah një aspekt të veçantë:

- **Rössler-i** thekson strukturën gjeometrike të një tërheqësi kaotik relativisht të thjeshtë.
- **Duffing-u** lidh kaosin me oshilatorët mekanikë jolinearë dhe me seksionet e Poincaré-së.
- **Hénon-i** tregon fuqinë e hartave diskrete në gjenerimin e tërheqësve fraktalë.
- **Harta standarde** paraqet tranzicionin drejt kaosit në dinamikën Hamiltoniane.
- **Qarku i Chua-s** jep një realizim eksperimental të drejtpërdrejtë të kaosit në elektronikë.

Në terma pedagogjikë, ky zgjerim është i rëndësishëm sepse e bën të qartë se **kaosi është një paradigmë e përgjithshme e sistemeve jolineare**, dhe jo thjesht një shembull i izoluar.

---

## 10. Eksperimente të lira

Qelizat më poshtë mund të përdoren për eksplorim të lirë nga studenti.


In [ ]:
# Eksperiment i lirë 1: Rössler
a, b, c = 0.2, 0.2, 5.7
t, sol = integrate_ode(rossler, np.array([0.2, 0.1, 0.0]), 0.0, 150.0, 0.02, a=a, b=b, c=c)
plt.figure(figsize=(7, 6))
plt.plot(sol[:,0], sol[:,1], linewidth=0.5)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Eksperiment i lirë: projekcioni (x,y) i Rössler-it")
plt.show()


In [ ]:
# Eksperiment i lirë 2: Duffing
params_duff = dict(delta=0.2, alpha=-1.0, beta=1.0, gamma=0.28, omega=1.2)
t, sol = integrate_ode(duffing, np.array([0.1, 0.0]), 0.0, 250.0, 0.01, **params_duff)
plt.figure(figsize=(8, 4.5))
plt.plot(t[t <= 80], sol[t <= 80, 0], linewidth=1.0)
plt.xlabel("Koha")
plt.ylabel("x(t)")
plt.title("Eksperiment i lirë: sinjali për Duffing-un")
plt.show()


In [ ]:
# Eksperiment i lirë 3: Hénon
xs, ys = iterate_henon(x0=0.12, y0=0.05, n=25000, a=1.4, b=0.3)
plt.figure(figsize=(7, 6))
plt.scatter(xs[2000:], ys[2000:], s=0.15)
plt.xlabel("$x_n$")
plt.ylabel("$y_n$")
plt.title("Eksperiment i lirë: harta e Hénon-it")
plt.show()


In [ ]:
# Eksperiment i lirë 4: Harta standarde
plt.figure(figsize=(8, 6))
for th0 in np.linspace(-np.pi, np.pi, 12):
    for p0 in np.linspace(-2.0, 2.0, 5):
        th, pp = standard_map(th0, p0, K=2.2, n=1000)
        plt.plot(th, pp, ".", markersize=0.6)
plt.xlabel(r"$\theta$")
plt.ylabel("p")
plt.title("Eksperiment i lirë: harta standarde për K = 2.2")
plt.show()


In [ ]:
# Eksperiment i lirë 5: Chua
params_chua = dict(alpha=15.6, beta=28.0, m0=-1.2, m1=-0.7)
t, sol = integrate_ode(chua, np.array([0.1, 0.0, 0.0]), 0.0, 220.0, 0.01, **params_chua)
plt.figure(figsize=(7, 6))
plt.plot(sol[:,0], sol[:,1], linewidth=0.4)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Eksperiment i lirë: projekcioni (x,y) i qarkut të Chua-s")
plt.show()
